# Study C Controllability Analysis

Legacy controllability notebook for Study C.

This notebook reads the original `ctrl_study_*` structured outputs.
Use the `v2` notebooks beside it for the same-case three-arm evaluation path.


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
STUDY_CODE = "C"
PRIMARY_LABEL = "CER / Controlled Entity Recall"


In [ ]:
def load_controllability_study_payload(study_code: str):
    study_file_map = {
        "A": "ctrl_study_a_results.json",
        "B": "ctrl_study_b_results.json",
        "C": "ctrl_study_c_results.json",
    }
    target_file = study_file_map[study_code]

    primary_rows = []
    profile_rows = []
    summary_rows = []

    if not RESULTS_DIR.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        study_path = model_dir / target_file
        if study_path.exists():
            payload = json.loads(study_path.read_text(encoding="utf-8"))
            primary_metric = payload.get("primary_metric", {})
            aggregate = payload.get("aggregate", {})
            primary_rows.append(
                {
                    "model": payload.get("model", model_dir.name),
                    "study": payload.get("study", study_code),
                    "primary_metric_name": primary_metric.get("metric_name"),
                    "primary_metric_value": primary_metric.get("value"),
                    "primary_ci_lower": primary_metric.get("ci_lower"),
                    "primary_ci_upper": primary_metric.get("ci_upper"),
                    "aggregate_score": aggregate.get("score"),
                    "aggregate_provisional_components": aggregate.get("provisional_components"),
                }
            )

            for entry in payload.get("controlled_profile", []):
                profile_rows.append(
                    {
                        "model": payload.get("model", model_dir.name),
                        "study": payload.get("study", study_code),
                        "metric_name": entry.get("metric_name"),
                        "classification": entry.get("classification"),
                        "controlled_value": entry.get("controlled_value"),
                        "baseline_value": entry.get("baseline_value"),
                        "delta_from_baseline": entry.get("delta_from_baseline"),
                        "control_score": entry.get("control_score"),
                        "included_in_rollup": entry.get("included_in_rollup"),
                        "notes": entry.get("notes"),
                    }
                )

        summary_path = model_dir / "controllability_summary.json"
        if summary_path.exists():
            summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))
            studies = summary_payload.get("studies", {})
            study_payload = studies.get(study_code)
            benchmark_payload = summary_payload.get("benchmark_control", {})
            if study_payload:
                summary_rows.append(
                    {
                        "model": summary_payload.get("model", model_dir.name),
                        "study": study_code,
                        "study_score": (study_payload.get("aggregate") or {}).get("score"),
                        "benchmark_control_score": benchmark_payload.get("score"),
                    }
                )

    return pd.DataFrame(primary_rows), pd.DataFrame(profile_rows), pd.DataFrame(summary_rows)


In [ ]:
primary_df, profile_df, summary_df = load_controllability_study_payload(STUDY_CODE)
display(primary_df.sort_values("primary_metric_value", ascending=False).reset_index(drop=True))
